# Three Frameworks, One Ontology, One Graph

**`SimpleKGPipeline`, `LLMGraphTransformer` and `PropertyGraphIndex` on the same ten documents — and what each of them hides.**

[Notebook 01](01_gliner25_knowledge_graphs.ipynb) built a knowledge graph by hand: ontology, extractor,
resolver, graph assembly. [Notebook 02](02_neo4j_graphrag_retrieval.ipynb) loaded it into Neo4j.
[Notebook 03](03_extractors_head_to_head.ipynb) measured the extractor against alternatives. That is four
components and a few hundred lines of `kgx`.

Three widely used libraries offer to collapse the whole thing into about six lines:

| | package | what it claims to do |
|---|---|---|
| **neo4j-graphrag** | `neo4j-graphrag` | `SimpleKGPipeline`: split, extract, resolve, write |
| **LangChain** | `langchain-neo4j` | `LLMGraphTransformer` + `Neo4jGraph.add_graph_documents` |
| **LlamaIndex** | `llama-index-core` + `llama-index-graph-stores-neo4j` | `PropertyGraphIndex.from_documents` |

This notebook runs all three on the same ten documents, with the same `BUSINESS_NEWS` ontology, driven by the
same Claude Haiku through the same disk cache, writing into the same Neo4j — and scores their output against
the same 47 gold triples notebook 03 used. The `kgx` pipeline from notebooks 01–03 is in the table as the
baseline.

The questions worth answering are not "which has the nicest API".

1. Can they take *your* ontology, or only a schema shaped the way they like it?
2. Do they need a function-calling model? (No. All three have a prompt-and-parse fallback, and finding it is
   most of the integration work.)
3. What do they actually write into Neo4j, and can you read it back out?
4. **Does any of them do entity resolution?** That is the through-line of this whole repo, and the answer is
   almost no.

### What you need

A Neo4j 5.26+ instance and the `claude` CLI — the same setup as notebooks 02 and 03, no API key. Every LLM
response is cached to `output/llm_cache/`, so a second run of this notebook is free and returns identical
numbers.

> **This notebook wipes the database.** Neo4j Community has exactly one database and all three frameworks
> write a `__Entity__` label, so they cannot coexist. Each framework runs against an empty graph, its output
> is read back into Python, and then it is deleted. The last section reloads notebook 02's graph, its indexes
> and its embeddings, so notebook 02 still works afterwards.

```bash
docker run -d --name extraction-sandbox-neo4j \
  -p 7690:7687 -p 7476:7474 \
  -e NEO4J_AUTH=neo4j/sandbox-kg \
  neo4j:5.26
```

## 0. Setup

In [1]:
import os, sys, time, json, warnings, subprocess
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 48)

import kgx
from kgx.data.documents import DOCUMENTS, GOLD_DOC_FACTS, ALIAS_GROUPS
from kgx.evaluate import score_triples, graph_triples, MatchPolicy
from kgx.llm import ClaudeCLI
from kgx import frameworks as fw

ONTOLOGY = kgx.BUSINESS_NEWS

print(f"corpus     {len(DOCUMENTS)} documents, {sum(len(d['text'].split()) for d in DOCUMENTS)} words")
print(f"ontology   {len(ONTOLOGY.entities)} entity types, {len(ONTOLOGY.relations)} relation types")
print(f"gold       {len(GOLD_DOC_FACTS)} triples, {len(ALIAS_GROUPS)} alias groups")

corpus     10 documents, 2284 words
ontology   13 entity types, 15 relation types
gold       47 triples, 13 alias groups


### The dependency situation, before anything else

Two of the three are not in this project's `pyproject.toml`, and one of them cannot be:

In [2]:
import re
from importlib.metadata import requires, version

# ^neo4j and not neo4j-graphrag: the requirement strings differ by one hyphen.
DRIVER_REQ = re.compile(r"^neo4j(?![-_A-Za-z])")

rows = []
for pkg in ["neo4j-graphrag", "langchain-neo4j", "llama-index-core",
            "llama-index-graph-stores-neo4j", "json-repair"]:
    pins = [r for r in (requires(pkg) or []) if DRIVER_REQ.match(r.split(";")[0].strip())]
    rows.append({"package": pkg, "installed": version(pkg),
                 "declares about the neo4j driver": pins[0] if pins else "—"})
display(pd.DataFrame(rows).set_index("package"))

print(f"neo4j driver actually installed: {version('neo4j')}")

,installed,declares about the neo4j driver
package,,
neo4j-graphrag,1.18.0,"neo4j<7.0.0,>=5.17.0"
langchain-neo4j,0.10.0,"neo4j<7.0.0,>=5.25.0"
llama-index-core,0.14.24,—
llama-index-graph-stores-neo4j,0.7.0,"neo4j<6,>=5.16.0"
json-repair,0.63.4,—


neo4j driver actually installed: 6.2.0


`llama-index-graph-stores-neo4j` declares `neo4j<6`, this project pins `neo4j>=6.2.0`, and `uv add` refuses
the combination outright:

```
× No solution found when resolving dependencies:
╰─▶ Because all versions of llama-index-graph-stores-neo4j depend on neo4j>=5.16.0,<6.0.0 and your
    project depends on llama-index-graph-stores-neo4j, we can conclude that your project depends on
    neo4j>=5.16.0,<6.0.0. And because your project depends on neo4j>=6.2.0 ... unsatisfiable.
```

Two ways out, both verified. Relax the project pin to `"neo4j>=5.28,<6"` and `uv add` all three — everything
below runs unchanged on driver 5.28.4, and `neo4j-graphrag` and `langchain-neo4j` both accept it. Or install
into the environment without the resolver's blessing:

```bash
uv pip install langchain-neo4j llama-index-core llama-index-graph-stores-neo4j json-repair
```

which is what produced the table above: the store works fine against driver 6.2.0 — constructor, APOC schema
refresh, `upsert_nodes`, index creation — the declared upper bound is simply conservative. The cost is that
the packages are not in the lockfile and a later `uv sync` removes them. Either choice is defensible; both
are worth knowing before you find out at import time.

`langchain-neo4j` supplies **both** `Neo4jGraph` and `LLMGraphTransformer`, so `langchain-experimental` is
not needed. `json-repair` is what LangChain's no-function-calling path parses with.

### Connect, and record what is already there

Notebook 02 left its graph in this database. Section 10 puts it back; this is the snapshot it will be checked
against.

In [3]:
import neo4j
from kgx.neo4j_io import Neo4jConfig, connect, counts

config = Neo4jConfig.from_env(
    uri=os.environ.get("NEO4J_URI", "bolt://localhost:7690"),
    user=os.environ.get("NEO4J_USER", "neo4j"),
    password=os.environ.get("NEO4J_PASSWORD", "sandbox-kg"),
)

if "driver" in globals():
    driver.close()
driver = connect(config, notifications="OFF")

records, _, _ = driver.execute_query(
    "CALL dbms.components() YIELD name, versions, edition RETURN name, versions[0] AS version, edition")
for r in records:
    print(f"{r['name']} {r['version']} ({r['edition']})")

BEFORE = counts(driver)
print(f"\nalready in {config.uri}: {BEFORE['nodes']} nodes, {BEFORE['relationships']} relationships")
print(f"  labels: {', '.join(sorted(BEFORE['by_label']))[:150]}")

Neo4j Kernel 5.26.19 (community)

already in bolt://localhost:7690: 119 nodes, 241 relationships
  labels: BusinessEvent, BusinessSegment, Commodity, Company, Document, FinancialMetric, Geography, Person, Product, Regulator, RiskFactor, Sector, Security, __


Community edition, one database. `CREATE DATABASE` is not a licensed command here, so the three frameworks
cannot be given a namespace each. Nor is a `source` property enough: neo4j-graphrag's entity resolver runs

```python
match_query = "MATCH (entity:__Entity__) "        # resolver.py:121, no scoping of any kind
```

and both of the other two also write `__Entity__`, so leaving one framework's nodes in place while another
runs would let the resolver merge across them. The only reliable isolation on one database is time:
`kgx.frameworks.wipe` between runs, and read the output back into Python before it is deleted.

## 1. One ontology, three dialects

Every framework wants the schema in its own shape and its own casing. The `Ontology` object from notebook 01
compiles to all three.

In [4]:
DIALECTS = fw.schema_dialects(ONTOLOGY)

display(pd.DataFrame({
    "kgx ontology": ONTOLOGY.entity_names,
    "neo4j-graphrag": [n["label"] for n in DIALECTS["graphrag"]["node_types"]],
    "LangChain":      DIALECTS["langchain"]["allowed_nodes"],
    "LlamaIndex":     DIALECTS["llamaindex"]["entities"],
}).set_index("kgx ontology").head(6))

print(f"legal (head, RELATION, tail) patterns: {len(DIALECTS['patterns'])}")
print(f"graphrag  schema keys : {sorted(DIALECTS['graphrag'])}")
print(f"langchain schema keys : {sorted(DIALECTS['langchain'])}")
print(f"llamaindex schema keys: {sorted(DIALECTS['llamaindex'])}")

,neo4j-graphrag,LangChain,LlamaIndex
kgx ontology,,,
company,Company,Company,COMPANY
person,Person,Person,PERSON
regulator,Regulator,Regulator,REGULATOR
product,Product,Product,PRODUCT
business_segment,BusinessSegment,BusinessSegment,BUSINESS_SEGMENT
geography,Geography,Geography,GEOGRAPHY


legal (head, RELATION, tail) patterns: 52
graphrag  schema keys : ['node_types', 'patterns', 'relationship_types']
langchain schema keys : ['allowed_nodes', 'allowed_relationships']
llamaindex schema keys: ['entities', 'relations', 'validation_schema']


`PascalCase` for two of them, `UPPER_SNAKE` for LlamaIndex. That last one is not a style preference — §5
shows what happens if you ignore it.

All three accept the endpoint constraints as well as the type lists, which is more than the raw prompt in
`kgx.llm.LLMExtractor` does. 52 legal patterns from 15 relations.

## 2. One LLM behind three interfaces

There is no API key here. The model is the `claude` CLI as a subprocess, wrapped in `kgx.llm.ClaudeCLI` with
a content-addressed disk cache. None of the three frameworks has an adapter for that, and all three
type-check the object you pass, so each needs a small subclass of a different base class.

`kgx.frameworks` builds all three around one `CallLog`, which is what makes the cost column below comparable:
the log records the cost and duration recorded on each response *when it was first made*, so a cache replay
still reports what the run really cost.

In [5]:
llm = ClaudeCLI(model="haiku", cache_dir=ROOT / "output" / "llm_cache")
print("claude CLI:", llm.check())
print(f"cache: {llm.cache_dir} ({len(list(llm.cache_dir.glob('*.json')))} responses)\n")

log = fw.CallLog(llm)
adapters = {
    "neo4j-graphrag": fw.graphrag_llm(log),
    "LangChain":      fw.langchain_llm(log),
    "LlamaIndex":     fw.llamaindex_llm(log),
}
for name, obj in adapters.items():
    base = type(obj).__mro__[1].__name__
    print(f"  {name:16} {type(obj).__name__:16} extends {base}")

claude CLI: 2.1.246 (Claude Code)
cache: /Users/lyonwj/github/johnymontana/extraction-sandbox/output/llm_cache (105 responses)



  neo4j-graphrag   ClaudeCLILLM     extends LLMBase
  LangChain        ClaudeCLIChat    extends BaseChatModel
  LlamaIndex       ClaudeCLILLM     extends CustomLLM


### None of them needs function calling

Every one of the three has two code paths: a structured-output path that binds a tool schema, and a
prompt-the-JSON-shape-and-parse-the-reply path. A subprocess CLI can only serve the second, and each library
picks the path from a different signal:

- **neo4j-graphrag** reads `llm.supports_structured_output` off the class. `False` selects the V1 prompt path,
  where the reply goes through `json_repair`.
- **LangChain** *probes* — it calls `llm.with_structured_output(...)` in the constructor and catches
  `NotImplementedError`. `BaseChatModel` raises exactly that when a subclass has not overridden `bind_tools`.
  So the fallback is automatic as long as you do not implement `bind_tools`.
- **LlamaIndex** reads `llm.metadata.is_function_calling_model`. `False` routes
  `astructured_predict` to `LLMTextCompletionProgram`, which appends the JSON schema to the prompt as text.

In [6]:
print(f"graphrag   supports_structured_output    = {adapters['neo4j-graphrag'].supports_structured_output}")
print(f"llamaindex metadata.is_function_calling  = {adapters['LlamaIndex'].metadata.is_function_calling_model}")

try:
    adapters["LangChain"].with_structured_output(dict)
    print("langchain  with_structured_output        -> no error (function-calling path!)")
except NotImplementedError as exc:
    print(f"langchain  with_structured_output        -> NotImplementedError: {exc}")

graphrag   supports_structured_output    = False
llamaindex metadata.is_function_calling  = False
langchain  with_structured_output        -> NotImplementedError: with_structured_output is not implemented for this model.


## 3. neo4j-graphrag — `SimpleKGPipeline`

The most complete of the three: it splits the text, extracts, **resolves**, and writes a lexical graph
(`Document`, `Chunk`, `FROM_CHUNK`) alongside the entities. Four things about the current signature are worth
knowing before you write the call:

- `embedder` is a **required positional argument** even if nothing is embedded. Omit it and you get
  `TypeError: __init__() missing 1 required positional argument`.
- There is **no `run()`**, only `run_async()`. `PipelineResult` has two fields, `run_id` and `result`, and
  `result` is the *last component's* output — the resolver's statistics, not the graph.
- The schema keys are `node_types` / `relationship_types` / `patterns`. `entities` / `relations` /
  `potential_schema` still work and emit a `DeprecationWarning`.
- `from_file=False` plus `run_async(text=...)` is how you feed it strings. `from_pdf` is a deprecated alias.

`perform_entity_resolution` defaults to `True`, so this is the only one of the three that ships a resolution
stage at all. What that stage actually does is the interesting part, and §8 measures it.

In [7]:
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.embeddings import SentenceTransformerEmbeddings

try:
    SimpleKGPipeline(llm=adapters["neo4j-graphrag"], driver=driver, schema=DIALECTS["graphrag"])
except TypeError as exc:
    print(f"without an embedder -> TypeError: {exc}")

print("\nwiping:", fw.wipe(driver))

embedder = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")   # local, no API key
pipeline = SimpleKGPipeline(
    llm=fw.graphrag_llm(log.reset()),
    driver=driver,
    embedder=embedder,
    schema=DIALECTS["graphrag"],
    from_file=False,
    perform_entity_resolution=True,
    neo4j_database=config.database,
)

async def run_graphrag():
    # One call per document; the extractor fans out to max_concurrency=5 inside.
    return [await pipeline.run_async(text=d["text"], document_metadata={"doc_id": d["doc_id"]})
            for d in DOCUMENTS]

started = time.time()
results = fw.run_coroutine(run_graphrag)   # asyncio.run() cannot start inside a Jupyter loop
graphrag_wall = time.time() - started

print(f"\n{len(results)} documents in {graphrag_wall:.1f}s wall")
print(f"last PipelineResult.result: {results[-1].result}")
print(f"LLM: {log.summary()}")

without an embedder -> TypeError: SimpleKGPipeline.__init__() missing 1 required positional argument: 'embedder'



wiping: {'nodes_deleted': 119, 'constraints_dropped': 14, 'indexes_dropped': 4}



10 documents in 0.6s wall
last PipelineResult.result: {'resolver': {'number_of_nodes_to_resolve': 110, 'number_of_created_nodes': 103}}
LLM: {'llm_calls': 10, 'cache_hits': 10, 'prompt_chars': 119327, 'reply_chars': 39147, 'cost_usd': 0.1615, 'model_seconds': 106.7}


`number_of_nodes_to_resolve` / `number_of_created_nodes` on that last run is the resolver's report over the
**whole database**, not over the last document. The gap between the two numbers is every merge it made.

In [8]:
graphrag_out = fw.read_back(driver, ONTOLOGY, "neo4j-graphrag")
graphrag_out.wall_seconds = graphrag_wall
graphrag_out.llm = log.summary()

print(f"{graphrag_out.n_nodes} nodes, {graphrag_out.n_relationships} relationships written")
print(f"  entity nodes    : {len(graphrag_out.entities)}")
print(f"  ontology triples: {len(graphrag_out.typed_triples)}")
print(f"  labels          : {sorted(graphrag_out.node_labels)}")
print(f"  lexical edges   : {graphrag_out.dropped_lexical}")

123 nodes, 320 relationships written
  entity nodes    : 103
  ontology triples: 141
  labels          : ['BusinessEvent', 'BusinessSegment', 'Chunk', 'Commodity', 'Company', 'Document', 'FinancialMetric', 'Geography', 'Litigation', 'Person', 'Product', 'Regulator', 'RiskFactor', 'Sector', 'Security', '__Entity__', '__KGBuilder__']
  lexical edges   : 179


Every node carries `__KGBuilder__`; extracted entities also carry `__Entity__`; `Document` and `Chunk` do
not. The lexical graph is more than half of the relationships — `FROM_CHUNK` from each entity to the chunk it
came from, plus `FROM_DOCUMENT`. That is provenance, and it is genuinely useful, but it is not knowledge and
the read-back in §6 excludes it.

## 4. LangChain — `LLMGraphTransformer`

The smallest surface of the three: a transformer that turns `Document` objects into `GraphDocument` objects,
and a separate `Neo4jGraph.add_graph_documents` that writes them. Extraction and storage are not coupled,
which is the nicest thing about it.

Two hard limits on the no-function-calling path, both enforced in the constructor:

In [9]:
from langchain_neo4j import LLMGraphTransformer, Neo4jGraph
from langchain_core.documents import Document

try:
    LLMGraphTransformer(llm=fw.langchain_llm(log), allowed_nodes=["Company"],
                        node_properties=True, ignore_tool_usage=True)
except ValueError as exc:
    print(f"node_properties=True -> ValueError: {exc}")

print("\nand the fallback prompt asks for a flat edge list:")
print('  {"head", "head_type", "relation", "tail", "tail_type"}')
print("  -> an entity mentioned in no relation is never written at all.")

node_properties=True -> ValueError: The 'node_properties' and 'relationship_properties' parameters cannot be used in combination with a LLM that doesn't support native function calling.

and the fallback prompt asks for a flat edge list:
  {"head", "head_type", "relation", "tail", "tail_type"}
  -> an entity mentioned in no relation is never written at all.


So on this environment LangChain extracts bare `(id, type)` nodes and nothing else. neo4j-graphrag and
LlamaIndex can both attach properties. It also does **no chunking** — one LLM call per `Document`, whatever
its length.

### Before running it: the prompt is not deterministic

`create_unstructured_prompt` builds the allowed-relationship line as

```python
rel_types_str = str(list({item[1] for item in rel_types}))    # llm.py:222 — a set
```

A `set` of strings, rendered straight into the prompt. Python randomises string hashing per process, so the
prompt text changes every time the interpreter restarts. Three subprocesses, same arguments:

In [10]:
PROBE = r"""
import re
from langchain_neo4j.graph_transformers.llm import create_unstructured_prompt
prompt = create_unstructured_prompt(
    ["Company", "Person"],
    [("Company", "ACQUIRES", "Company"), ("Person", "OFFICER_OF", "Company"),
     ("Company", "SUPPLIES", "Company"), ("Company", "COMPETES_WITH", "Company")],
    "tuple")
system = prompt.messages[0]
text = getattr(system, "content", None) or system.prompt.template
print(re.search(r"\['[A-Z_]+'(, '[A-Z_]+')*\]", text).group(0))
"""
for i in range(3):
    out = subprocess.run([sys.executable, "-c", PROBE], capture_output=True, text=True)
    print(f"  process {i}: {out.stdout.strip() or out.stderr.strip()[-90:]}")

  process 0: ['ACQUIRES', 'COMPETES_WITH', 'SUPPLIES', 'OFFICER_OF']


  process 1: ['OFFICER_OF', 'SUPPLIES', 'COMPETES_WITH', 'ACQUIRES']


  process 2: ['COMPETES_WITH', 'ACQUIRES', 'OFFICER_OF', 'SUPPLIES']


That is a real reproducibility bug in a framework sold on convenience: the same code, the same schema and the
same corpus produce a different prompt, a different extraction and a different content-addressed cache key on
every run. `kgx.frameworks.stabilise_prompt` sorts that one list back into a stable order at the adapter
boundary — reordering an enumeration does not change what the prompt asks for, and it is the difference
between a benchmark that replays from cache and one that re-bills every time.

Sorting is not free of consequences, though, and §4.2 measures what the reordering costs.

In [11]:
print("wiping:", fw.wipe(driver))

transformer = LLMGraphTransformer(
    llm=fw.langchain_llm(log.reset()),                 # stabilise=True by default
    allowed_nodes=DIALECTS["langchain"]["allowed_nodes"],
    allowed_relationships=DIALECTS["langchain"]["allowed_relationships"],
    ignore_tool_usage=True,
    strict_mode=True,
)
print(f"_function_call={transformer._function_call}   "
      f"_relationship_type={transformer._relationship_type!r}")

lc_docs = [Document(page_content=d["text"], metadata={"doc_id": d["doc_id"], "title": d["title"]})
           for d in DOCUMENTS]

started = time.time()
graph_documents = transformer.convert_to_graph_documents(lc_docs)
langchain_wall = time.time() - started

print(f"\n{len(DOCUMENTS)} documents in {langchain_wall:.1f}s wall")
print(f"  {sum(len(g.nodes) for g in graph_documents)} nodes, "
      f"{sum(len(g.relationships) for g in graph_documents)} relationships")
print(f"LLM: {log.summary()}")

wiping: {'nodes_deleted': 123, 'constraints_dropped': 0, 'indexes_dropped': 1}
_function_call=False   _relationship_type='tuple'

10 documents in 0.0s wall
  118 nodes, 113 relationships
LLM: {'llm_calls': 10, 'cache_hits': 10, 'prompt_chars': 141187, 'reply_chars': 19820, 'cost_usd': 0.184, 'model_seconds': 60.1}


`_relationship_type='tuple'` means `strict_mode` filters on the full `(source_type, RELATION, target_type)`
pattern rather than on the relation name alone. That is the strongest ontology enforcement of the three, and
because the prompt does not depend on `strict_mode`, its effect can be measured for free by re-parsing the
same cached responses with the filter off.

### 4.1 What `strict_mode` actually caught

In [12]:
loose = LLMGraphTransformer(
    llm=fw.langchain_llm(fw.CallLog(llm)),             # same prompts -> all cache hits
    allowed_nodes=DIALECTS["langchain"]["allowed_nodes"],
    allowed_relationships=DIALECTS["langchain"]["allowed_relationships"],
    ignore_tool_usage=True, strict_mode=False,
)
unfiltered = loose.convert_to_graph_documents(lc_docs)

def edge_set(gdocs):
    return [(r.source.type, r.source.id, r.type, r.target.type, r.target.id)
            for g in gdocs for r in g.relationships]

from collections import Counter
kept, raw = Counter(edge_set(graph_documents)), Counter(edge_set(unfiltered))
print(f"model produced {sum(raw.values())} relations; strict_mode kept {sum(kept.values())}\n")
for (ht, h, rel, tt, t) in (raw - kept):
    print(f"  dropped ({ht}) {h} -{rel}-> ({tt}) {t}")
    print(f"           ontology declares {rel.lower()}: "
          f"{'|'.join(ONTOLOGY.relation(rel.lower()).head)} -> "
          f"{'|'.join(ONTOLOGY.relation(rel.lower()).tail)}")

model produced 115 relations; strict_mode kept 113

  dropped (Company) Northwind Logistics Inc. -OPERATES_IN-> (BusinessSegment) Freight Brokerage
           ontology declares operates_in: company|business_segment -> geography|sector
  dropped (BusinessSegment) Freight Brokerage -FACES_RISK-> (RiskFactor) softening demand for expedited freight
           ontology declares faces_risk: company|sector -> risk_factor


Two edges out of the model's whole output. The model was already almost perfectly compliant, which is worth
noticing: with the ontology in the prompt *and* enforced afterwards, the filter is a safety net rather than
the mechanism.

Note the second one. `business_segment -faces_risk-> risk_factor` is a fact the text supports and the
*ontology* forbids — `faces_risk` is declared with `company|sector` heads. The filter enforces your model
including its gaps, which is an argument for keeping the discards visible rather than dropping them silently.

### 4.2 What the prompt reordering costs

One document, two stable orderings of the same 15-item list, everything else identical.

In [13]:
one_doc = [Document(page_content=DOCUMENTS[0]["text"], metadata={"doc_id": "d01"})]
orderings = {}
for order in ("sorted", "reversed"):
    probe = LLMGraphTransformer(
        llm=fw.langchain_llm(fw.CallLog(llm), order=order),
        allowed_nodes=DIALECTS["langchain"]["allowed_nodes"],
        allowed_relationships=DIALECTS["langchain"]["allowed_relationships"],
        ignore_tool_usage=True, strict_mode=True)
    g = probe.convert_to_graph_documents(one_doc)[0]
    orderings[order] = {(r.source.id, r.type, r.target.id) for r in g.relationships}
    print(f"  {order:9} -> {len(g.nodes)} nodes, {len(orderings[order])} relations")

only_rev = orderings["reversed"] - orderings["sorted"]
only_srt = orderings["sorted"] - orderings["reversed"]
print(f"\nshared {len(orderings['sorted'] & orderings['reversed'])}, "
      f"only sorted {len(only_srt)}, only reversed {len(only_rev)}")
for h, r, t in sorted(only_rev | only_srt):
    print(f"    {h} -{r}-> {t}")

  sorted    -> 8 nodes, 8 relations
  reversed  -> 9 nodes, 9 relations

shared 8, only sorted 0, only reversed 1
    Northwind Logistics Inc. -REPORTS_METRIC-> adjusted earnings per share


Reordering an enumeration in the prompt changed the extraction. Every number in this notebook is a sample of
one from a distribution that includes that variance, and so is every framework benchmark you have read.

### 4.3 Write it

In [14]:
lc_graph = Neo4jGraph(url=config.uri, username=config.user, password=config.password,
                      refresh_schema=False)
lc_graph.add_graph_documents(graph_documents, include_source=True, baseEntityLabel=True)

langchain_out = fw.read_back(driver, ONTOLOGY, "LangChain")
langchain_out.wall_seconds = langchain_wall
langchain_out.llm = log.summary()

print(f"{langchain_out.n_nodes} nodes, {langchain_out.n_relationships} relationships written")
print(f"  entity nodes    : {len(langchain_out.entities)}")
print(f"  ontology triples: {len(langchain_out.typed_triples)}")
print(f"  labels          : {sorted(langchain_out.node_labels)}")
print(f"  lexical edges   : {langchain_out.dropped_lexical}")

81 nodes, 216 relationships written
  entity nodes    : 71
  ontology triples: 98
  labels          : ['BusinessEvent', 'BusinessSegment', 'Commodity', 'Company', 'Document', 'FinancialMetric', 'Geography', 'Litigation', 'Person', 'Product', 'Regulator', 'RiskFactor', 'Sector', 'Security', '__Entity__']
  lexical edges   : 118


`baseEntityLabel=True` adds `__Entity__` and a uniqueness constraint on `__Entity__.id`; `include_source=True`
adds `(:Document)-[:MENTIONS]->(entity)`. Note the key property: **`id`**, not `name`. That single difference
is most of what §6's normalisation has to absorb.

## 5. LlamaIndex — `PropertyGraphIndex`

`PropertyGraphIndex.from_documents` runs a list of `kg_extractors` over parsed nodes and writes to a
`PropertyGraphStore`. Two extractors are relevant:

- `SimpleLLMPathExtractor` — ontology-free by construction: its signature accepts no type lists at all, so
  every node label and relation name is whatever the model happened to write. Nothing to compare against a
  fixed ontology.
- `SchemaLLMPathExtractor` — takes `possible_entities` / `possible_relations` as `Literal` types and a
  `kg_validation_schema` of `(head, REL, tail)` triples. This is the one to use.

`embed_kg_nodes=False` matters: the default is `True` and would demand an embedding model that, without an
OpenAI key, does not exist here.

### The gotcha that costs an afternoon

`SchemaLLMPathExtractor`'s triplet validator normalises before it validates:

```python
triplet[key]["type"] = triplet[key]["type"].replace(" ", "_").upper()
_ = triplet_cls(**triplet)          # validates against your Literal[...]
except (KeyError, ValueError):
    continue                        # <- silently drops the triplet
```

Give it `Literal["Company", "Person"]`, the model returns `"Company"` exactly as asked, the validator rewrites
it to `"COMPANY"`, the `Literal` rejects it, and the triplet is dropped **with no error and no warning**.
`raise_on_error=True` does not help, because nothing raises. The visible symptom is a perfect LLM response
and an empty graph.

In [15]:
from typing import Literal
from llama_index.core import Document as LlamaDocument, PropertyGraphIndex
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

# No LLM call is made here -- just the schema class the validator is built from.
payload = lambda labels: {"triplets": [{
    "subject":  {"name": "Priya Raman", "type": labels[1]},
    "relation": {"type": "OFFICER_OF"},
    "object":   {"name": "Northwind Logistics Inc.", "type": labels[0]},
}]}

for labels in (("Company", "Person"), ("COMPANY", "PERSON")):
    probe = SchemaLLMPathExtractor(
        llm=adapters["LlamaIndex"],
        possible_entities=Literal[labels], possible_relations=Literal[("OFFICER_OF",)],
        kg_validation_schema=[(labels[1], "OFFICER_OF", labels[0])], strict=True)
    kept = probe.kg_schema_cls(**payload(labels)).triplets
    print(f"  possible_entities=Literal{labels}  ->  {len(kept)} triplet(s) kept")

  possible_entities=Literal('Company', 'Person')  ->  0 triplet(s) kept
  possible_entities=Literal('COMPANY', 'PERSON')  ->  1 triplet(s) kept


Same model output, same schema, one casing difference, everything or nothing. LlamaIndex's own
`DEFAULT_ENTITIES` are upper case, so the library is self-consistent — it just does not tell you.
Relationship labels are `UPPER_SNAKE` already and are unaffected.

In [16]:
print("wiping:", fw.wipe(driver))

li_llm = fw.llamaindex_llm(log.reset())
extractor = SchemaLLMPathExtractor(
    llm=li_llm,
    possible_entities=Literal[tuple(DIALECTS["llamaindex"]["entities"])],
    possible_relations=Literal[tuple(DIALECTS["llamaindex"]["relations"])],
    kg_validation_schema=DIALECTS["llamaindex"]["validation_schema"],
    # LlamaIndex is the only one of the three with a per-chunk triplet budget.
    # It does not truncate -- the value is interpolated into the prompt ("Try to
    # limit to the output 25 extracted paths") and the model mostly honours it --
    # but it is a knob the other two frameworks do not expose, so "same corpus,
    # same ontology, same model" holds with this one asterisk.
    strict=True, max_triplets_per_chunk=25, num_workers=4,
)
store = Neo4jPropertyGraphStore(username=config.user, password=config.password,
                                url=config.uri, database=config.database, refresh_schema=False)

li_docs = [LlamaDocument(text=d["text"], metadata={"doc_id": d["doc_id"]}) for d in DOCUMENTS]

started = time.time()
# _insert_nodes calls asyncio.run() unconditionally, which a Jupyter kernel's
# running loop rejects -- so hand the whole call to a thread that has no loop.
index = fw.run_off_loop(lambda: PropertyGraphIndex.from_documents(
    li_docs, llm=li_llm, kg_extractors=[extractor],
    property_graph_store=store, embed_kg_nodes=False, show_progress=False,
))
llamaindex_wall = time.time() - started

print(f"\n{len(DOCUMENTS)} documents in {llamaindex_wall:.1f}s wall")
print(f"LLM: {log.summary()}")

wiping: {'nodes_deleted': 81, 'constraints_dropped': 1, 'indexes_dropped': 0}



10 documents in 0.5s wall
LLM: {'llm_calls': 10, 'cache_hits': 10, 'prompt_chars': 36547, 'reply_chars': 39292, 'cost_usd': 0.0991, 'model_seconds': 100.4}


In [17]:
llamaindex_out = fw.read_back(driver, ONTOLOGY, "LlamaIndex")
llamaindex_out.wall_seconds = llamaindex_wall
llamaindex_out.llm = log.summary()

print(f"{llamaindex_out.n_nodes} nodes, {llamaindex_out.n_relationships} relationships written")
print(f"  entity nodes    : {len(llamaindex_out.entities)}")
print(f"  ontology triples: {len(llamaindex_out.typed_triples)}")
print(f"  labels          : {sorted(llamaindex_out.node_labels)}")
print(f"  lexical edges   : {llamaindex_out.dropped_lexical}")

122 nodes, 271 relationships written
  entity nodes    : 112
  ontology triples: 119
  labels          : ['BUSINESS_EVENT', 'BUSINESS_SEGMENT', 'COMMODITY', 'COMPANY', 'Chunk', 'FINANCIAL_METRIC', 'GEOGRAPHY', 'LITIGATION', 'PERSON', 'PRODUCT', 'REGULATOR', 'RISK_FACTOR', 'SECTOR', '__Entity__', '__Node__']
  lexical edges   : 152


`__Node__` on everything, `__Entity__` on the extracted entities, `Chunk` on the text, key property `name`.
A third schema, a third set of marker labels, and a third answer to "what is the node's identity".

## 6. Reading three schemas back into one shape

Three frameworks wrote three different graphs into the same database. Before any of them can be scored
against the same 47 gold triples, they have to be read back into one comparable form — and **an unfair
normalisation invalidates the entire comparison**, so it is worth stating exactly what
`kgx.frameworks.read_back` does.

| | key property | entity marker | lexical nodes | provenance edge | label casing |
|---|---|---|---|---|---|
| neo4j-graphrag | `name` | `__Entity__` + `__KGBuilder__` | `Document`, `Chunk` | `FROM_CHUNK`, `FROM_DOCUMENT` | `PascalCase` |
| LangChain | **`id`** | `__Entity__` | `Document` | `MENTIONS` | as the model returned it |
| LlamaIndex | `name` | `__Entity__` + `__Node__` | `Chunk` | `MENTIONS` | `UPPER_SNAKE` |

Three rules, in order:

1. A node is an **entity** if any of its labels maps to an ontology entity type after case and separators are
   stripped (`BUSINESS_SEGMENT`, `BusinessSegment` and `business_segment` all key to `businesssegment`). That
   rule excludes `Document`, `Chunk`, `__Entity__`, `__Node__` and `__KGBuilder__` without naming any of them
   — none is an ontology type.
2. Its name is `n.name` if present, else `n.id`.
3. An edge counts if both endpoints are entities and its type maps to an ontology relation. Provenance edges
   are counted, not silently discarded.

What `read_back` deliberately does **not** do: normalise, deduplicate or resolve names. Whatever the
framework stored is what goes to the scorer. That is the point of §8.

In [18]:
OUTPUTS = {o.framework: o for o in (graphrag_out, langchain_out, llamaindex_out)}

display(pd.DataFrame([{
    "framework": name,
    "nodes written": o.n_nodes,
    "relationships written": o.n_relationships,
    "entity nodes": len(o.entities),
    "ontology triples": len(o.typed_triples),
    "provenance edges": o.dropped_lexical,
    "unmapped rel types": sum(o.dropped_unmapped_rel.values()),
    "unnamed entities": o.unnamed_entities,
} for name, o in OUTPUTS.items()]).set_index("framework"))

,nodes written,relationships written,entity nodes,ontology triples,provenance edges,unmapped rel types,unnamed entities
framework,,,,,,,
neo4j-graphrag,123,320,103,141,179,0,0
LangChain,81,216,71,98,118,0,0
LlamaIndex,122,271,112,119,152,0,0


In [19]:
# Three casings, one ontology: fold each framework's labels back onto the ontology types.
by_type = []
for etype in ONTOLOGY.entity_names:
    row = {"ontology type": etype}
    for name, o in OUTPUTS.items():
        row[name] = sum(n for label, n in o.node_labels.items()
                        if fw.canonical_type(label, ONTOLOGY) == etype)
    by_type.append(row)
display(pd.DataFrame(by_type).set_index("ontology type").T)

for name, o in OUTPUTS.items():
    other = sorted(l for l in o.node_labels if fw.canonical_type(l, ONTOLOGY) is None)
    print(f"  {name:16} non-ontology labels: {', '.join(other)}")

ontology type,company,person,regulator,product,business_segment,geography,sector,security,financial_metric,risk_factor,litigation,business_event,commodity
neo4j-graphrag,10,7,3,9,10,20,2,2,18,6,3,11,2
LangChain,9,6,3,4,3,16,1,1,19,3,1,2,3
LlamaIndex,9,7,1,7,7,20,1,0,37,12,1,8,2


  neo4j-graphrag   non-ontology labels: Chunk, Document, __Entity__, __KGBuilder__
  LangChain        non-ontology labels: Document, __Entity__
  LlamaIndex       non-ontology labels: Chunk, __Entity__, __Node__


Same ontology, three casings, and quite different behaviour underneath it. The `financial_metric` column is
the one to look at: LlamaIndex made roughly twice as many metric nodes as either of the others (37 against 19 and 18), which is an
extraction-behaviour difference no amount of schema enforcement can see. §7 shows where that lands.

### Did they respect the ontology?

Three ways to break it: invent a node label, invent a relationship type, or connect two legal types with a
relation that does not allow them.

In [20]:
display(pd.DataFrame([{"framework": name, **o.violations(ONTOLOGY)}
                      for name, o in OUTPUTS.items()]).set_index("framework"))

for name, o in OUTPUTS.items():
    bad = o.illegal_edges(ONTOLOGY)
    if bad:
        print(f"{name}: {len(bad)} illegal endpoint(s), e.g. "
              f"({bad[0][1]}) {bad[0][0]} -{bad[0][2]}-> ({bad[0][4]}) {bad[0][3]}")

,off_ontology_labels,off_ontology_relations,illegal_endpoints
framework,,,
neo4j-graphrag,0,0,0
LangChain,0,0,0
LlamaIndex,0,0,0


Zero across the board. Every framework's ontology enforcement is a post-filter over the model's reply rather
than a decoding constraint — the mechanism notebook 03 contrasts with GLiNER's typed decoding — but on this
corpus the post-filter is airtight, and §4.1 showed how little it had to catch.

Worth keeping in proportion: a post-filter guarantees the graph is *well-typed*, not that it is *right*. It
cannot tell you about the edge the model never produced, and it turns a mis-typed fact into a missing one.

## 7. The scoreboard

Two baselines from earlier notebooks, scored the same way:

- **GLiNER2.5 + `kgx`** — the graph notebook 01 built and notebook 02 loaded. A 194M encoder, no LLM, plus
  this repo's blocking/scoring/clustering resolver.
- **Claude Haiku + `kgx`** — notebook 03's LLM path: the *same model* the three frameworks are driving, with
  the ontology rendered into a single prompt by `kgx.llm.LLMExtractor`, then the same resolver and graph
  assembly.

The second baseline is the interesting control. Same model, same ontology, same corpus — the only difference
from the frameworks is who wrote the pipeline.

In [21]:
from kgx.graph import KnowledgeGraph
from kgx.llm import LLMExtractor

gliner_kg = KnowledgeGraph.from_json(ROOT / "output" / "business_news_kg.json", ONTOLOGY)
gliner_triples = graph_triples(gliner_kg)
gliner_entities = [(e.canonical, e.type) for e in gliner_kg.entities.values()]
print(f"GLiNER2.5 graph (from notebook 01): {len(gliner_kg.entities)} entities, "
      f"{len(gliner_triples)} triples")

baseline_log = fw.CallLog(llm)
doc_graphs = LLMExtractor(baseline_log).extract_batch(DOCUMENTS, ONTOLOGY, workers=4)
mentions = [m for g in doc_graphs for m in g.mentions]
resolution = kgx.EntityResolver(threshold=0.90).learn_aliases(d["text"] for d in DOCUMENTS).resolve(mentions)
kgx_llm_kg = kgx.build_graph(doc_graphs, resolution, ONTOLOGY)
kgx_llm_triples = graph_triples(kgx_llm_kg)
kgx_llm_entities = [(e.canonical, e.type) for e in kgx_llm_kg.entities.values()]

print(f"Claude Haiku + kgx: {len(mentions)} mentions -> {len(kgx_llm_kg.entities)} entities, "
      f"{len(kgx_llm_triples)} triples")
print(f"  ontology violations kept: {sum(len(g.validate(ONTOLOGY)) for g in doc_graphs)}")
print(f"  {baseline_log.summary()}")

GLiNER2.5 graph (from notebook 01): 109 entities, 68 triples


Claude Haiku + kgx: 165 mentions -> 89 entities, 100 triples
  ontology violations kept: 14
  {'llm_calls': 10, 'cache_hits': 10, 'prompt_chars': 50787, 'reply_chars': 23372, 'cost_usd': 0.0748, 'model_seconds': 60.0}


Two things in that output are worth pausing on.

The `kgx` LLM path **kept** ontology violations where all three frameworks filtered theirs out. `LLMExtractor`
drops unknown type names but does not check endpoint types, so illegal edges survive into the graph. That is
a point for the frameworks, and a bug report for this repo.

The cost and duration on those baseline calls are **not** comparable with the framework numbers in §9. They
were cached by a different session under a different `claude` CLI invocation — divide by ten and it is 6.0
seconds and 0.75 cents a call, against 6.0–10.7 seconds and 1.0–1.8 cents for the framework runs. The vintage
is the problem rather than the magnitude: a
content-addressed cache replays the old metadata along with the old text. Cached benchmarks are reproducible,
not immortal: the responses are still the right responses, and the meter readings attached to them are from
whatever the code did that day.

### Matching policy

The same two policies notebook 03 uses. `strict` requires the predicted name to normalise to the gold
canonical name; `normalised + aliases` also accepts any surface form in the corpus alias table, so `Cascade`
counts as `Cascade Freight Systems`. Both are reported, because the gap between them is the finding.

In [22]:
STRICT  = MatchPolicy(normalize_names=True, use_aliases=False)
LENIENT = MatchPolicy(normalize_names=True, use_aliases=True)

SYSTEMS = {
    "GLiNER2.5 + kgx (nb 01)":    dict(triples=gliner_triples,  entities=gliner_entities),
    "Claude Haiku + kgx (nb 03)": dict(triples=kgx_llm_triples, entities=kgx_llm_entities),
    "neo4j-graphrag":             dict(triples=graphrag_out.triples,   entities=graphrag_out.entities),
    "LangChain":                  dict(triples=langchain_out.triples,  entities=langchain_out.entities),
    "LlamaIndex":                 dict(triples=llamaindex_out.triples, entities=llamaindex_out.entities),
}

rows = []
for name, sysinfo in SYSTEMS.items():
    scored = {}
    for policy in (STRICT, LENIENT):
        s = score_triples(sysinfo["triples"], GOLD_DOC_FACTS, policy=policy, aliases=ALIAS_GROUPS)
        scored[policy.label] = s
    sysinfo["scores"] = scored
    rows.append({
        "system": name,
        "triples": len(sysinfo["triples"]),
        "strict P": round(scored["strict"].precision, 3),
        "strict R": round(scored["strict"].recall, 3),
        "strict F1": round(scored["strict"].f1, 3),
        "alias P": round(scored["normalised + aliases"].precision, 3),
        "alias R": round(scored["normalised + aliases"].recall, 3),
        "alias F1": round(scored["normalised + aliases"].f1, 3),
        "F1 gap": round(scored["normalised + aliases"].f1 - scored["strict"].f1, 3),
    })

scoreboard = pd.DataFrame(rows).set_index("system")
display(scoreboard.sort_values("alias F1", ascending=False))

,triples,strict P,strict R,strict F1,alias P,alias R,alias F1,F1 gap
system,,,,,,,,
LangChain,98,0.316,0.660,0.428,0.316,0.660,0.428,0.000
Claude Haiku + kgx (nb 03),100,0.300,0.638,0.408,0.300,0.638,0.408,0.000
GLiNER2.5 + kgx (nb 01),68,0.279,0.404,0.330,0.279,0.404,0.330,0.000
LlamaIndex,119,0.227,0.574,0.325,0.227,0.574,0.325,0.000
neo4j-graphrag,141,0.199,0.596,0.298,0.206,0.617,0.309,0.011


Read the `F1 gap` column first, and then read what it failed to measure.

It was meant to show how much of each system's score is supplied by the *evaluator's* alias table rather than
by the system — the entity resolution the scorer performs on a pipeline's behalf. The prediction was that the
two `kgx` pipelines would barely move and the three frameworks would gain several times as much.

Four of the five rows are **0.000**. Only `neo4j-graphrag` moves at all, by 0.011.

The column is measuring nothing because the alias table it credits (`ALIAS_GROUPS`) maps the *canonical*
surface of thirteen entities to their variants, and the scorer's normaliser already strips the differences
that separate most predicted names from those canonical forms — legal suffixes, determiners, case. A system
that writes `Northwind Logistics Inc.` where gold says `Northwind Logistics Inc.` gains nothing from an alias
table, whether or not it ever resolved anything.

So the honest reading is narrower than the one this cell was built to deliver: **on this corpus the strict and
alias-credit policies are nearly the same measurement**, and the question of how much resolution each
framework skipped has to be answered directly — by rewriting their names and re-scoring, which §8 does.

In [23]:
for name in ("LangChain", "neo4j-graphrag"):
    print(f"══ {name}")
    print(SYSTEMS[name]["scores"]["normalised + aliases"].report(limit=5))
    print()

══ LangChain
precision 0.316  recall 0.660  f1 0.428   (31/47 gold, 98 predicted, policy: normalised + aliases)

  missed (16):
    Northwind Logistics Inc. -supplies-> Halcyon Semiconductor Corporation
    Torrent Microsystems -produces-> controller boards
    Northwind Logistics Inc. -reports_metric-> FY2025 revenue of $2.41 billion
    Northwind Logistics Inc. -reports_metric-> FY2025 adjusted operating margin of 8.7%
    Northwind Logistics Inc. -reports_metric-> FY2025 adjusted EPS of $3.12

  spurious (67):
    Cascade Freight Systems -reports_metric-> revenue of $612 million in fiscal 2025
    Northwind Logistics Inc. -reports_metric-> $85 million of annual run-rate synergies by 2028
    Surface Transportation Board -participant_in-> formal review of Northwind's proposed purchase of Cascade
    Northwind Logistics -participant_in-> formal review of Northwind's proposed purchase of Cascade
    Northwind Logistics -acquires-> Cascade

══ neo4j-graphrag
precision 0.206  recall 0.61

The `spurious` lists are dominated by facts that are true and simply not in the answer key — 47 hand-written
triples are a floor on what these documents contain, not a ceiling. Precision here should be read as
"agreement with a partial answer key", and only recall is a clean measurement.

## 8. Entity resolution, or the lack of it

This is the through-line of the whole repo, so state it plainly:

| | entity resolution | what it actually is |
|---|---|---|
| **neo4j-graphrag** | `perform_entity_resolution=True` by default | `SinglePropertyExactMatchResolver`: merge nodes with the **same label and byte-identical `name`** |
| **LangChain** | none | `add_graph_documents` MERGEs on `id`, which dedupes exact strings at write time |
| **LlamaIndex** | none | `upsert_nodes` MERGEs on the node's id |
| **`kgx`** | `EntityResolver` | normalise, block, score (fuzzy + embedding + context), cluster, canonicalise |

Only one of the three ships anything, and what it ships is exact string equality. Exact string equality is
what `MERGE` already gives you. So the honest summary is that **none of these frameworks resolves entities**;
two of them deduplicate and one of them deduplicates twice.

Here is what neo4j-graphrag's resolver did on this corpus:

In [24]:
stats = results[-1].result["resolver"]
print(f"nodes seen: {stats['number_of_nodes_to_resolve']} -> "
      f"{stats['number_of_created_nodes']} after exact-match merging "
      f"({stats['number_of_nodes_to_resolve'] - stats['number_of_created_nodes']} merges)")

# What survived: every stored node whose name falls in a corpus alias group.
report = fw.duplication_report(graphrag_out.entities, ALIAS_GROUPS)
for canonical, variants in sorted(report["detail"].items()):
    print(f"\n  {canonical}")
    print(f"    stored as {len(variants)} separate nodes: {', '.join(variants)}")

nodes seen: 110 -> 103 after exact-match merging (7 merges)

  Cascade Freight Systems
    stored as 2 separate nodes: Cascade, Cascade Freight Systems

  Halcyon Semiconductor Corporation
    stored as 4 separate nodes: HLCN, Halcyon, Halcyon Semiconductor, Halcyon Semiconductor Corporation

  Northwind Logistics Inc.
    stored as 3 separate nodes: NWL, Northwind Logistics, Northwind Logistics Inc.


Exact matching cannot merge `NWL` into `Northwind Logistics Inc.`, and there is no configuration of
`SimpleKGPipeline` that would — it exposes `perform_entity_resolution` as a boolean and no `resolver=`
argument at all. To use a different resolver you have to abandon `SimpleKGPipeline` and assemble
`neo4j_graphrag.experimental.pipeline` components yourself, at which point most of the convenience is gone.

### How much duplication is left

`ALIAS_GROUPS` is the corpus's hand-written clustering of every surface form onto a canonical name. Any two
stored nodes that fall in the same group are two nodes for one real thing.

In [25]:
dup_rows = []
for name, sysinfo in SYSTEMS.items():
    r = fw.duplication_report(sysinfo["entities"], ALIAS_GROUPS)
    sysinfo["duplication"] = r
    dup_rows.append({
        "system": name,
        "entity nodes": r["nodes"],
        "nodes covered by alias table": r["nodes_in_alias_table"],
        "real entities behind them": r["real_entities_covered"],
        "entities split across nodes": r["split_entities"],
        "surplus nodes": r["surplus_nodes"],
    })
display(pd.DataFrame(dup_rows).set_index("system"))

,entity nodes,nodes covered by alias table,real entities behind them,entities split across nodes,surplus nodes
system,,,,,
GLiNER2.5 + kgx (nb 01),109,14,13,1,1
Claude Haiku + kgx (nb 03),89,16,13,3,3
neo4j-graphrag,103,19,13,3,6
LangChain,71,17,13,3,4
LlamaIndex,112,14,11,3,3


**That table does not say what it was supposed to say.** Three of the frameworks leave three entities split
across a handful of surplus nodes — and so does the `kgx` LLM pipeline, which has a resolver. Only the GLiNER
pipeline gets down to one. Node-count duplication turns out to be a weak discriminator, and the honest
version of the claim is narrower than "the frameworks do not resolve and the `kgx` pipelines do".

Look at what is left over in the two resolved pipelines:

In [26]:
def with_types(sysinfo):
    types = {}
    for name, etype in sysinfo["entities"]:
        types.setdefault(name, etype)
    return {canonical: [(v, types.get(v, "?")) for v in variants]
            for canonical, variants in sysinfo["duplication"]["detail"].items()}

for name in ("GLiNER2.5 + kgx (nb 01)", "Claude Haiku + kgx (nb 03)"):
    print(f"══ {name}")
    for canonical, variants in sorted(with_types(SYSTEMS[name]).items()):
        print(f"   {canonical}")
        print(f"     {', '.join(f'{v} ({t})' for v, t in variants)}")
    print()

══ GLiNER2.5 + kgx (nb 01)
   Halcyon Semiconductor Corporation
     HLCN (security), Halcyon Semiconductor Corporation (company)

══ Claude Haiku + kgx (nb 03)
   Cascade Freight Systems
     Cascade Freight (company), Cascade Freight Systems (company)
   Halcyon Semiconductor Corporation
     HLCN (security), Halcyon Semiconductor Corporation (company)
   Northwind Logistics Inc.
     NWL (security), Northwind Logistics Inc (company)



Two different failures there, and the type column separates them. `HLCN` and `NWL` were typed as a `security`
rather than as the company, and `EntityResolver` only ever scores pairs of the **same ontology type**, so
those were never compared with the company at all — notebook 02 §5.5 hit the same node from the retrieval
side. `Halcyon` and `Cascade Freight` are plain same-type misses below the 0.90 threshold. A resolver is not a
solved problem either. The difference is that it is a stage you own, so both of those have a fix.

What separates the resolved pipelines from the unresolved ones is the other half of resolution:
**canonicalisation** — choosing one name per cluster and putting it on the node. That is what the `F1 gap`
column in §7 measures and what duplicate-node counting misses. A framework that only ever saw
`Halcyon Semiconductor` stores exactly that: one node, no duplication, and a strict match against
`Halcyon Semiconductor Corporation` that fails anyway.

### What the missing stage is worth

`kgx.EntityResolver` over each framework's stored node names — no re-extraction, no LLM calls, a local MiniLM
and about 60 lines of blocking and scoring. Then re-score the same triples under the **strict** policy, where
the evaluator gives no alias credit at all.

In [27]:
corpus = [d["text"] for d in DOCUMENTS]
er_rows = []

for name, out in OUTPUTS.items():
    mapping = fw.resolve_names(out.entities, corpus_texts=corpus)
    renamed = sum(1 for stored, canon in mapping.items() if stored != canon)
    before = SYSTEMS[name]["scores"]["strict"]
    after = score_triples(out.canonicalised(mapping), GOLD_DOC_FACTS,
                          policy=STRICT, aliases=ALIAS_GROUPS)
    SYSTEMS[name]["strict_after_er"] = after
    # canonicalised() rewrites endpoints but does not collapse the duplicates that
    # rewriting creates, so `after` can only ever gain matched triples. Count the
    # collapse separately -- it is what a real resolver would also do to precision.
    canon_triples = out.canonicalised(mapping)
    collapsed = len(canon_triples) - len(set(canon_triples))
    er_rows.append({
        "framework": name,
        "names": len(mapping),
        "canonical after ER": len(set(mapping.values())),
        "names rewritten": renamed,
        "dupe triples created": collapsed,
        "strict F1": round(before.f1, 3),
        "strict F1 + kgx resolver": round(after.f1, 3),
        "alias-credit F1 (ceiling)": round(SYSTEMS[name]["scores"]["normalised + aliases"].f1, 3),
    })

display(pd.DataFrame(er_rows).set_index("framework"))

,names,canonical after ER,names rewritten,dupe triples created,strict F1,strict F1 + kgx resolver,alias-credit F1 (ceiling)
framework,,,,,,,
neo4j-graphrag,103,96,9,16,0.298,0.309,0.309
LangChain,71,67,7,13,0.428,0.428,0.428
LlamaIndex,112,108,11,9,0.325,0.325,0.325


That table does not pay off either. Two of the three frameworks had no gap to close — their strict and
alias-credit F1 are identical — so the resolver rewrote 7 and 11 names and moved their scores by nothing.
Only `neo4j-graphrag` had a gap, and the resolver closed it exactly: 0.298 to 0.309, which is its
alias-credit ceiling, for no model calls and no re-extraction.

One caveat on how that column was computed: `canonicalised()` rewrites the endpoints but does not collapse
the duplicate triples rewriting creates, so this measurement can only *gain* matched triples — it never pays
the precision cost a real resolver would. The `dupe triples created` column is that unpaid bill.

That is a much weaker result than the section was built to show, and it is worth stating plainly rather than
rounding up. The stage all three frameworks leave out is still the cheapest in the pipeline and still the one
that decides whether the graph can be queried by name — but on ten documents with thirteen gold entities,
skipping it costs almost nothing measurable, because the scorer's own normaliser is doing the work a resolver
would have done.

That is what a "complete" KG construction framework is quietly not doing for you. Everything upstream of it —
chunking, prompting, schema enforcement, writing to Neo4j — is the part that is easy to write and easy to
demo. Resolution is the part that is neither.

## 9. What each run cost

`wall` is the time this notebook took, which on a warm cache is the time to parse and write, not to think.
`model seconds` is the sum of the durations recorded on each response when it was **first** produced, and
`cost` likewise — that is what the run costs the first time, and it is the number to compare.

In [28]:
cost = pd.DataFrame([{
    "framework": name,
    "LLM calls": o.llm["llm_calls"],
    "prompt chars": o.llm["prompt_chars"],
    "reply chars": o.llm["reply_chars"],
    "model seconds": o.llm["model_seconds"],
    "cost USD": o.llm["cost_usd"],
    "wall seconds (cached)": round(o.wall_seconds, 1),
    "triples": len(o.typed_triples),
} for name, o in OUTPUTS.items()]).set_index("framework")
display(cost)

print(f"total spend for the three framework runs: "
      f"${sum(o.llm['cost_usd'] for o in OUTPUTS.values()):.3f}")
print(f"cache now holds {len(list(llm.cache_dir.glob('*.json')))} responses; "
      f"delete {llm.cache_dir} to force real calls.")

,LLM calls,prompt chars,reply chars,model seconds,cost USD,wall seconds (cached),triples
framework,,,,,,,
neo4j-graphrag,10,119327,39147,106.7,0.1615,0.6,141
LangChain,10,141187,19820,60.1,0.1840,0.0,98
LlamaIndex,10,36547,39292,100.4,0.0991,0.5,119


total spend for the three framework runs: $0.445
cache now holds 105 responses; delete /Users/lyonwj/github/johnymontana/extraction-sandbox/output/llm_cache to force real calls.


Ten documents, ten calls each — none of the three chunked this corpus into more than one call per document,
so the chunking machinery two of them ship was never exercised. What differs is the *prompt*. LlamaIndex
sends the schema as a compact JSON schema; LangChain writes the full 52-pattern list into both the system and
the human message of every call. Divide the `prompt chars` column by ten for the per-call figure and the
ordering of the cost column follows it exactly.

That matters more than it looks, because the schema is overhead you pay on every chunk of every document
forever. A 13-type, 15-relation ontology is small. At 40 types the prompt is the bill.

## 10. The comparison

In [29]:
def kg_violations(kg):
    """Illegal (head_type, relation, tail_type) edges in an assembled kgx graph."""
    return sum(1 for e in kg.edges
               if not ONTOLOGY.permits(kg.entity(e.head).type, e.type, kg.entity(e.tail).type))

VIOLATIONS = {name: sum(o.violations(ONTOLOGY).values()) for name, o in OUTPUTS.items()}
VIOLATIONS["GLiNER2.5 + kgx (nb 01)"] = kg_violations(gliner_kg)
VIOLATIONS["Claude Haiku + kgx (nb 03)"] = kg_violations(kgx_llm_kg)

OWN_ER = {"neo4j-graphrag": "exact string match", "LangChain": "none", "LlamaIndex": "none",
          "GLiNER2.5 + kgx (nb 01)": "kgx resolver", "Claude Haiku + kgx (nb 03)": "kgx resolver"}

final = pd.DataFrame([{
    "system": name,
    "entities": len(sysinfo["entities"]),
    "triples": len(sysinfo["triples"]),
    "strict F1": round(sysinfo["scores"]["strict"].f1, 3),
    "alias F1": round(sysinfo["scores"]["normalised + aliases"].f1, 3),
    "alias recall": round(sysinfo["scores"]["normalised + aliases"].recall, 3),
    "surplus nodes": sysinfo["duplication"]["surplus_nodes"],
    "ontology violations": VIOLATIONS[name],
    "cost USD": OUTPUTS[name].llm["cost_usd"] if name in OUTPUTS else float("nan"),
    "own ER": OWN_ER[name],
} for name, sysinfo in SYSTEMS.items()]).set_index("system")

display(final.sort_values("alias F1", ascending=False))

,entities,triples,strict F1,alias F1,alias recall,surplus nodes,ontology violations,cost USD,own ER
system,,,,,,,,,
LangChain,71,98,0.428,0.428,0.660,4,0,0.1840,none
Claude Haiku + kgx (nb 03),89,100,0.408,0.408,0.638,3,13,NaN,kgx resolver
GLiNER2.5 + kgx (nb 01),109,68,0.330,0.330,0.404,1,0,NaN,kgx resolver
LlamaIndex,112,119,0.325,0.325,0.574,3,0,0.0991,none
neo4j-graphrag,103,141,0.298,0.309,0.617,6,0,0.1615,exact string match


### What the numbers say

**The matching policy changed nothing here.** Sorted by strict F1 the table is in exactly the order it is in
under alias-credit matching — LangChain, Claude Haiku + `kgx`, GLiNER2.5 + `kgx`, LlamaIndex,
`neo4j-graphrag`. Only `neo4j-graphrag` moves at all, by 0.011, and it moves without changing places. That is
a null result for a column §6 was built to interrogate, and §6 explains why: the scorer's normaliser already
absorbs most of what the alias table would have credited. It is worth knowing that the two policies agree on
this corpus rather than assuming they would disagree elsewhere.

**The spread among the three frameworks is wider than any gap between a framework and a hand-built
pipeline.** Whatever the row order, the two `kgx` pipelines sit in the middle of the three framework rows,
not above or below them. Choosing a framework is not trading quality for convenience — it is choosing one
prompt and one filter over another, and on 47 gold triples those are not distinguishable with any
confidence. §4.2 moved one document's output by one relation with a reordered list.

**The LLM pipelines find much more than the encoder.** Alias recall runs 0.57–0.66 across the four LLM
systems against 0.40 for `GLiNER2.5 + kgx`, and two of the four pay for it in precision — LlamaIndex at 0.227
and `neo4j-graphrag` at 0.206, against the encoder's 0.279. The same trade notebook 03 measured, reproduced
through three different wrappers around the same Haiku. LangChain is the exception: it beats the encoder on
precision *and* recall, and it does it by producing the fewest triples of any LLM pipeline here.

**Ontology enforcement is real and it is a post-filter.** All three respected the 13 types, 15 relations and
52 patterns exactly, because all three drop what does not fit. That is genuinely useful and it is not the
same guarantee as GLiNER's typed decoding in notebook 03: a decoding constraint changes which graph the model
picks, a post-filter deletes the parts of the answer that did not fit. §4.1 shows a true fact being deleted
because the *ontology* was too narrow.

**Nobody resolves entities.** One framework merges byte-identical names, which `MERGE` already does. Bolting
this repo's resolver onto their stored names afterwards rewrote 9, 7 and 11 names for zero model calls — and
moved strict F1 on exactly one of the three, `neo4j-graphrag`, from 0.298 to 0.309. The other two were
already at their alias-credit ceiling, so there was no gap for a resolver to close on this corpus. The
missing stage is real; what §8 could not show is that it costs them anything measurable here. If you take one thing from this notebook: the framework gets you to a graph, and then you still have
to write the stage that makes it queryable by name — and §8 shows that stage is not finished in this repo
either.

### When is a framework worth it

Use **neo4j-graphrag** when you are already on Neo4j and want the lexical graph — chunks, documents, and
`FROM_CHUNK` edges wired up for retrieval — without writing the writer. It is the most complete of the three
and the closest to notebook 02's shape. It is also the most opinionated: one resolver, taken or left.

Use **LangChain's `LLMGraphTransformer`** when you want extraction only. It is the one component here that
does not also own storage, so it drops into an existing pipeline, and `strict_mode` with `(head, REL, tail)`
tuples filters on edge direction as well as on type. Do not use it if you need node properties without a
function-calling model, and do not trust its prompt to be stable across processes.

Use **LlamaIndex's `PropertyGraphIndex`** when the graph is a retrieval index rather than a database — it is
built to be queried through its own retrievers, and the graph store is one of several backends. It sends the
leanest prompts. Budget an afternoon for the upper-case validator.

**Assemble it yourself** when the ontology is unusual, when you need entity resolution (which is to say:
always, eventually), when extraction has to be auditable edge by edge, or when the corpus is large enough
that one LLM call per document is the wrong economics. Notebook 03's escalation curve is that argument in
one chart.

### What each one hides

- **neo4j-graphrag** hides the resolver. `perform_entity_resolution=True` reads like a solved problem and is
  exact string matching over the entire database, unscoped, with no way to substitute your own.
- **LangChain** hides a `set` in the prompt. Nothing in the API suggests the extraction is
  process-dependent, and a benchmark built on it is not reproducible unless you go looking.
- **LlamaIndex** hides a silent drop. A validator that uppercases your labels and then rejects them for not
  being uppercase, with `continue` instead of a warning, produces an empty graph and a clean exit code.

All three hide the same larger thing: the graph they hand you has one node per surface string.

## 11. Putting notebook 02's graph back

The last framework's output is still in the database. Restore what was there when this notebook started:
the GLiNER graph, the source documents, the four search indexes and the embeddings behind them. After this
cell, notebook 02's retrievers work again without re-running notebook 02.

In [30]:
from kgx.neo4j_io import load_graph, load_documents, ENTITY_LABEL
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index, upsert_vectors
from neo4j_graphrag.types import EntityType

print("wiping the last framework's output:", fw.wipe(driver))

load_stats = load_graph(driver, gliner_kg)
doc_stats = load_documents(driver, DOCUMENTS, gliner_kg)
print(f"restored: {load_stats}, {doc_stats}")

INDEXES = {
    "document_vec": dict(kind="vector",   label="Document",   prop="embedding"),
    "document_ft":  dict(kind="fulltext", label="Document",   props=["title", "text"]),
    "entity_vec":   dict(kind="vector",   label=ENTITY_LABEL, prop="embedding"),
    "entity_ft":    dict(kind="fulltext", label=ENTITY_LABEL, props=["name", "aliases"]),
}
DIM = len(embedder.embed_query("dimension probe"))
for name, spec in INDEXES.items():
    if spec["kind"] == "vector":
        create_vector_index(driver, name, label=spec["label"], embedding_property=spec["prop"],
                            dimensions=DIM, similarity_fn="cosine")
    else:
        create_fulltext_index(driver, name, label=spec["label"], node_properties=spec["props"])

for query, label in [
    ("MATCH (d:Document) RETURN elementId(d) AS eid, d.title + '\n' + d.text AS text", "documents"),
    ("""MATCH (n:__Entity__)
        RETURN elementId(n) AS eid,
               n.type + ': ' + n.name +
               CASE WHEN size(n.aliases) > 0
                    THEN ' (also known as ' + reduce(s = '', a IN n.aliases | s + a + ', ') + ')'
                    ELSE '' END AS text""", "entities"),
]:
    records, _, _ = driver.execute_query(query, routing_=neo4j.RoutingControl.READ)
    upsert_vectors(driver, ids=[r["eid"] for r in records], embedding_property="embedding",
                   embeddings=[embedder.embed_query(r["text"]) for r in records],
                   entity_type=EntityType.NODE)
    print(f"  embedded {len(records)} {label}")

after = counts(driver)
print(f"\ndatabase at the start of this notebook: "
      f"{BEFORE['nodes']} nodes, {BEFORE['relationships']} relationships")
print(f"database now                          : "
      f"{after['nodes']} nodes, {after['relationships']} relationships")
if (after["nodes"], after["relationships"]) != (BEFORE["nodes"], BEFORE["relationships"]):
    print("  (they differ, so this database did not start out holding notebook 02's graph)")

wiping the last framework's output: {'nodes_deleted': 122, 'constraints_dropped': 2, 'indexes_dropped': 1}


restored: {'nodes': 109, 'relationships': 68, 'method': 'dynamic'}, {'documents': 10, 'mention_links': 173}


  embedded 10 documents


  embedded 109 entities

database at the start of this notebook: 119 nodes, 241 relationships
database now                          : 119 nodes, 241 relationships


## Where to take it

**A fair fight on chunking.** Every document here fits in one LLM call, so the chunkers two of these
frameworks ship never ran. On a corpus of 30-page filings the chunk boundary decides which facts are even
expressible, and that is where `SimpleKGPipeline`'s lexical graph starts to earn its keep.

**Resolution as a plugin.** §8 bolts `kgx.EntityResolver` onto framework output after the fact. The better
shape is a resolver that runs *against the live graph* — `kgx.CanonicalRegistry` over Neo4j — so that
document eleven merges into the entities documents one through ten created, rather than everything being
re-resolved in a batch. Notebook 02's closing section makes the same point about write-back.

**The variance question.** §4.2 changed one enumeration's order and the extraction moved. Nobody publishing
framework comparisons runs them five times. Five seeds across all three frameworks would cost a few dollars
and would tell you whether any of the differences in §10 exist at all.

**Retrieval, not F1.** These graphs were scored on triples. The question that matters is whether a retriever
can answer a question over them — and the entity-resolution gap hurts retrieval far more than it hurts a
triple score, because `HybridCypherRetriever` looking up `NWL` needs an alias list that none of these
frameworks produced. Notebook 02 §5.5 is the experiment; running it against these three graphs is the
follow-up.

**The other corpus.** Nothing here touched the agent-memory graph, which is where resolution, coreference and
temporal supersession all matter more than they do in news copy. None of these frameworks has anything to say
about a fact that stops being true.

In [31]:
driver.close()
print("driver closed")

driver closed
